# Helper Functions in HW 5

In [22]:
using LinearAlgebra
# Using defined functions in ECON8208Tools.jl
include("../ECON8208Tools.jl")
using .ECON8208Tools


In [23]:
# -------------------------------------------------------
# Initialize parameters
# -------------------------------------------------------

beta = 0.96          # discount factor
psi = 1.50          # utility weight on leisure
sigma = 2.00          # coefficient of relative risk aversion
gamma_n = 0.01        # population growth rate
gamma_z = 0.02        # labor-augmenting technology growth rate
theta = 0.36          # capital share in production
delta = 0.08          # depreciation rate
rho = 0.90          # persistence of productivity shock
sigma_e = 0.02        # standard deviation of productivity shock

# -------------------------------------------------------
# Derived growth objects after detrending
# -------------------------------------------------------

G = (1.0 + gamma_n) * (1.0 + gamma_z)
beta_tilde = beta * (1.0 + gamma_n) * (1.0 + gamma_z)^(1.0 - sigma)

# -------------------------------------------------------
# Additional settings for computation
# -------------------------------------------------------

# problem (a)
n_k = 1500         # number of grid points for capital
m_k = 4.0          # upper bound multiplier for capital
n_z = 5           # number of grid points for productivity
tol = 1e-6        # convergence tolerance
max_iter = 1000   # maximum number of iterations

# problem (b)
hstep_lq = 1e-6    # step size for numerical derivatives in LQ approximation

# -------------------------------------------------------
# Initial productivity state
# -------------------------------------------------------

z_ss = 1.0        # steady-state productivity
logz_ss = 0.0     # steady-state log productivity

# -------------------------------------------------------
# Initial capital condition
# Will be updated after computing steady-state capital
# -------------------------------------------------------

k0 = nothing


### Numerical approximation of the standard normal CDF

Tauchen's method requires repeated evaluation of the standard normal cumulative distribution function
$$
\Phi(x) = \int_{-\infty}^x \frac{1}{\sqrt{2\pi}} e^{-t^2/2} dt.
$$

Since $\Phi(x)$ does not have a closed-form expression in elementary functions, we approximate it numerically.

A common approximation writes $\Phi(x)$ as a function of the standard normal density
$$
\phi(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2},
$$
and a polynomial in
$$
t = \frac{1}{1 + p |x|},
$$
where $p$ is a constant.

For $x \ge 0$, the approximation is
$$
\Phi(x)
\approx
1 - \phi(x)\left(b_1 t + b_2 t^2 + b_3 t^3 + b_4 t^4 + b_5 t^5\right),
$$
where
$$
b_1 = 0.319381530,\quad
b_2 = -0.356563782,\quad
b_3 = 1.781477937,\quad
b_4 = -1.821255978,\quad
b_5 = 1.330274429,
$$
and
$$
p = 0.2316419.
$$

For $x < 0$, we use the symmetry property of the standard normal distribution:
$$
\Phi(x) = 1 - \Phi(-x).
$$

### Pseudocode for numerical approximation of $\Phi(x)$

**Input:** $x$

**Output:** $\Phi(x)$

**Algorithm:**

1. Set the approximation constants $b_1, b_2, b_3, b_4, b_5$, and $p$.
2. Compute
   $$
   t = \frac{1}{1 + p|x|}.
   $$
3. Compute the standard normal density
   $$
   \phi(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}.
   $$
4. Compute the approximation for the positive case:
   $$
   \Phi_+(x)
   =
   1 - \phi(x)\left(b_1 t + b_2 t^2 + b_3 t^3 + b_4 t^4 + b_5 t^5\right).
   $$
5. If $x \ge 0$, return $\Phi_+(x)$.
6. Otherwise, return
   $$
   1 - \Phi_+(x).
   $$



In [24]:
# -------------------------------------------------------
# Standard normal cumulative distribution function
# Numerical approximation without external packages
# Input:
#   x : scalar
# Output:
#   Phi(x)
# -------------------------------------------------------
function normcdf(x)
    b1 = 0.319381530
    b2 = -0.356563782
    b3 = 1.781477937
    b4 = -1.821255978
    b5 = 1.330274429
    p = 0.2316419

    t = 1.0 / (1.0 + p * abs(x))
    pdf = exp(-0.5 * x^2) / sqrt(2.0 * pi)

    cdf_pos = 1.0 - pdf * (b1 * t + b2 * t^2 + b3 * t^3 + b4 * t^4 + b5 * t^5)

    if x >= 0.0
        return cdf_pos
    else
        return 1.0 - cdf_pos
    end
end


normcdf (generic function with 1 method)

Test the numerical approximation of $\Phi(x)$ against a standard normal cumulative distribution function to ensure accuracy.

Below I used Distributions.jl's `cdf` function to compute the exact value of $\Phi(x)$ for a standard normal distribution, and compared it with the approximation defined above. 

The results show that the approximation is very close to the exact values, with small absolute errors across a range of $x$ values from -4 to 4.

However, I won't use Distributions package in the actual implementation.

In [25]:
# -------------------------------------------------------
# Test the numerical approximation of normcdf
# Input:
#   x : scalar
# Output:
#   Print a table comparing normcdf with Distributions.jl's cdf for a range of x values
# -------------------------------------------------------

# Just for testing, I want to compare defined normcdf with Distributions.jl's cdf, make sure they are close enough
using Distributions

x_test = collect(range(-4.0, 4.0, length=17))

println("      x        approx        exact        abs error")
println("------------------------------------------------------")

for x in x_test
    approx_val = normcdf(x)
    exact_val = cdf(Normal(), x)
    err = abs(approx_val - exact_val)

    println(rpad(round(x, digits=4), 10),
            lpad(round(approx_val, digits=8), 14),
            lpad(round(exact_val, digits=8), 14),
            lpad(round(err, digits=10), 16))
end


      x        approx        exact        abs error
------------------------------------------------------
-4.0            3.169e-5      3.167e-5         1.48e-8
-3.5          0.00023267    0.00023263         4.43e-8
-3.0          0.00134997     0.0013499         6.92e-8
-2.5          0.00620968    0.00620967         1.45e-8
-2.0          0.02275006    0.02275013         6.99e-8
-1.5          0.06680723     0.0668072         2.75e-8
-1.0          0.15865526    0.15865525          5.6e-9
-0.5          0.30853753    0.30853754          6.5e-9
0.0                  0.5           0.5         5.0e-10
0.5           0.69146247    0.69146246          6.5e-9
1.0           0.84134474    0.84134475          5.6e-9
1.5           0.93319277     0.9331928         2.75e-8
2.0           0.97724994    0.97724987         6.99e-8
2.5           0.99379032    0.99379033         1.45e-8
3.0           0.99865003     0.9986501         6.92e-8
3.5           0.99976733    0.99976737         4.43e-8
4.0          

### Tauchen discretization for AR(1) process

$z_t$ follows an AR(1) process. To solve the model, we need to discretize the state space of $z_t$ using Tauchen's method.

Suppose the productivity shock satisfies
$$
\log z_t = \rho \log z_{t-1} + \epsilon_t,
\qquad
\epsilon_t \sim N(0,\sigma_\epsilon^2).
$$

Let
$$
y_t = \log z_t.
$$
Then the process becomes
$$
y_t = \rho y_{t-1} + \epsilon_t.
$$

The unconditional standard deviation of $y_t$ is
$$
\sigma_y = \frac{\sigma_\epsilon}{\sqrt{1-\rho^2}}.
$$

Tauchen's method approximates this continuous AR(1) process by a finite-state Markov chain.  
First, we construct an evenly spaced grid for $y_t$ over the interval
$$
[-m\sigma_y,\; m\sigma_y],
$$
where $m$ is typically chosen as 3.

Let the grid points be
$$
y_1, y_2, \dots, y_{n_z},
$$
with step size
$$
\Delta = y_{j+1} - y_j.
$$

Then, for each current state $y_i$, we compute the transition probability to each future state $y_j$ using the normal distribution implied by
$$
y_{t+1} \mid y_t = y_i \sim N(\rho y_i,\sigma_\epsilon^2).
$$

For interior grid points,
$$
P_{ij}
=
\Pr\left(
y_j - \frac{\Delta}{2}
\le y_{t+1}
\le
y_j + \frac{\Delta}{2}
\;\middle|\;
y_t = y_i
\right).
$$

For the lower and upper boundary points,
$$
P_{i1}
=
\Pr\left(
y_{t+1}
\le
y_1 + \frac{\Delta}{2}
\;\middle|\;
y_t = y_i
\right),
$$
and
$$
P_{in_z}
=
\Pr\left(
y_{t+1}
\ge
y_{n_z} - \frac{\Delta}{2}
\;\middle|\;
y_t = y_i
\right).
$$

Finally, we exponentiate the grid to recover the productivity levels:
$$
z_j = e^{y_j}, \qquad j=1,\dots,n_z.
$$

### Pseudocode for Tauchen discretization

**Input:** $\rho, \sigma_\epsilon, n_z, m$

**Output:** $z$-grid, transition matrix $P$

**Algorithm:**

1. Compute the unconditional standard deviation
   $$
   \sigma_y = \frac{\sigma_\epsilon}{\sqrt{1-\rho^2}}.
   $$
2. Construct an evenly spaced grid for $y = \log z$ over
   $$
   [-m\sigma_y,\; m\sigma_y].
   $$
3. Compute the grid step size $\Delta$.
4. For each current grid point $y_i$:
   - For each future grid point $y_j$:
     - If $j=1$, compute the lower-tail transition probability.
     - If $j=n_z$, compute the upper-tail transition probability.
     - Otherwise, compute the probability of moving into the interval
       $$
       \left[y_j-\frac{\Delta}{2},\; y_j+\frac{\Delta}{2}\right].
       $$
5. Form the transition matrix $P$.
6. Set
   $$
   z_j = e^{y_j}
   $$
   for each grid point.
7. Return the $z$-grid and the transition matrix $P$.

In [26]:
# -------------------------------------------------------
# Tauchen discretization for log z_t = rho * log z_{t-1} + eps_t
# Input:
#   rho      : persistence parameter
#   sigma_e  : standard deviation of innovation
#   n_z      : number of grid points
#   m        : width parameter for Tauchen grid (default: 3.0)
# Output:
#   z_grid   : productivity grid in levels
#   Pz       : Markov transition matrix
# -------------------------------------------------------
function tauchen(rho, sigma_e, n_z; m=3.0)
    sigma_z = sigma_e / sqrt(1.0 - rho^2)
    logz_max = m * sigma_z
    logz_min = -m * sigma_z

    logz_grid = collect(range(logz_min, logz_max, length=n_z))
    step = logz_grid[2] - logz_grid[1]

    Pz = zeros(n_z, n_z)

    for i in 1:n_z
        for j in 1:n_z
            if j == 1
                cutoff = (logz_grid[1] - rho * logz_grid[i] + step / 2.0) / sigma_e
                Pz[i, j] = normcdf(cutoff)
            elseif j == n_z
                cutoff = (logz_grid[n_z] - rho * logz_grid[i] - step / 2.0) / sigma_e
                Pz[i, j] = 1.0 - normcdf(cutoff)
            else
                upper = (logz_grid[j] - rho * logz_grid[i] + step / 2.0) / sigma_e
                lower = (logz_grid[j] - rho * logz_grid[i] - step / 2.0) / sigma_e
                Pz[i, j] = normcdf(upper) - normcdf(lower)
            end
        end
    end

    z_grid = exp.(logz_grid)

    return z_grid, Pz
end

tauchen (generic function with 1 method)

In [27]:
# --------------------------------------------------------
# Test the Tauchen discretization function
# Input parameters for testing:
#   rho = 0.90
#   sigma_e = 0.02
#   n_z = 5
# Output:
#   z_grid : productivity grid in levels
#   Pz     : Markov transition matrix
# Verify that each row of Pz sums to 1
# --------------------------------------------------------

# Construct productivity grid and transition matrix
z_grid, Pz = tauchen(rho, sigma_e, n_z)

println("Productivity grid (z_grid):")
println(z_grid)
println("\nTransition matrix (Pz):")
println(Pz)

# Verify that each row of Pz sums to 1
row_sums = sum(Pz, dims=2)
println("\nRow sums of Pz (should be 1):")
println(row_sums)

Productivity grid (z_grid):
[0.8714041173542129, 0.9334902877664089, 1.0, 1.071248424440206, 1.1475731868656236]

Transition matrix (Pz):
[0.8490507598420803 0.1509453911710562 3.848986862231563e-6 1.2212453270876722e-15 0.0; 0.019473661313098134 0.8961919730286212 0.08433363870120059 7.269570799772751e-7 1.1102230246251565e-16; 1.2248331016095904e-7 0.0426599233154199 0.9146799084025399 0.0426599233154199 1.2248331016095904e-7; 1.1102230246251565e-16 7.269570799772751e-7 0.08433363870120059 0.8961919730286212 0.019473661313098134; 0.0 1.2212453270876722e-15 3.848986862231563e-6 0.1509453911710562 0.8490507598420803]

Row sums of Pz (should be 1):
[1.0; 1.0; 1.0; 1.0; 1.0;;]


## (a) Solve the model using value function iteration.

We can rewrite the detrended household problem as the following Bellman equation:
$$
\begin{aligned}
V(\tilde{k},z)
=
\max_{\tilde{k}',\,\tilde{h}}
\quad
\Biggl\{
&
\frac{
\left[
\tilde{c}(1-\tilde{h})^\psi
\right]^{1-\sigma}
}{1-\sigma}
+ \tilde{\beta} E\bigl[V(\tilde{k}',z')\bigr]
\Biggr\}, \\
\text{s.t.} \quad
\log z' &= \rho \log z + \epsilon',
\qquad
\epsilon' \sim N(0,\sigma_\epsilon^2), \\
\tilde{c} &= \tilde{k}^\theta (z \tilde{h})^{1-\theta} - (1+\gamma_n)(1+\gamma_z)\tilde{k}' + (1-\delta)\tilde{k}, \\
\tilde{k}' &\ge 0, \\
0 &\le \tilde{h} \le 1, \\
\tilde{c} &> 0.
\end{aligned}
$$

The computational burden of brute-force value function iteration is of order
$$
n_k \times n_z \times n_h \times n_k,
$$
which is large. However, the choice of $\tilde{h}$ is a static problem. Hence, for each given $(\tilde{k}, z, \tilde{k}')$, we can first solve for the optimal labor choice $\tilde{h}$ and then substitute it back into the Bellman equation.

Given $\tilde{k}, z, \tilde{k}'$, the household solves
$$
\begin{aligned}
\max_{\tilde{h}}
\quad
&
\frac{
\left[
\tilde{c}(1-\tilde{h})^\psi
\right]^{1-\sigma}
}{1-\sigma} \\
\text{s.t.} \quad
\tilde{c} &= \tilde{k}^\theta (z \tilde{h})^{1-\theta} - (1+\gamma_n)(1+\gamma_z)\tilde{k}' + (1-\delta)\tilde{k}, \\
\tilde{c} &> 0, \\
0 &\le \tilde{h} \le 1.
\end{aligned}
$$

The first-order condition for $\tilde{h}$ is
$$
\frac{(1-\theta)\tilde{k}^{\theta} z^{1-\theta}\tilde{h}^{-\theta}}
{\tilde{k}^{\theta}(z\tilde{h})^{1-\theta} - (1+\gamma_n)(1+\gamma_z)\tilde{k}' + (1-\delta)\tilde{k}}
=
\frac{\psi}{1-\tilde{h}}.
$$

Solving this equation gives the optimal labor supply
$$
\tilde{h}^*(\tilde{k}, z, \tilde{k}').
$$

Substituting $\tilde{h}^*$ back into the Bellman equation, we obtain
$$
\begin{aligned}
V(\tilde{k},z)
=
\max_{\tilde{k}'}
\quad
\Biggl\{
&
\frac{
\left[
\tilde{c}^*(1-\tilde{h}^*(\tilde{k}, z, \tilde{k}'))^\psi
\right]^{1-\sigma}
}{1-\sigma}
+ \tilde{\beta} E\bigl[V(\tilde{k}',z')\bigr]
\Biggr\}, \\
\text{s.t.} \quad
\log z' &= \rho \log z + \epsilon',
\qquad
\epsilon' \sim N(0,\sigma_\epsilon^2), \\
\tilde{c}^*
&=
\tilde{k}^\theta \bigl(z \tilde{h}^*(\tilde{k}, z, \tilde{k}')\bigr)^{1-\theta}
-
(1+\gamma_n)(1+\gamma_z)\tilde{k}'
+
(1-\delta)\tilde{k}, \\
\tilde{k}' &\ge 0, \\
\tilde{c}^* &> 0.
\end{aligned}
$$


### Solving steady state

The steady-state values $(\tilde{k}^{ss}, \tilde{h}^{ss}, \tilde{c}^{ss})$ are determined by the following system:
$$
\begin{aligned}
\tilde{c}^{ss}
&=
(\tilde{k}^{ss})^\theta (\tilde{h}^{ss})^{1-\theta}
-
\bigl((1+\gamma_n)(1+\gamma_z)-(1-\delta)\bigr)\tilde{k}^{ss}, \\
\frac{1}{\tilde{\beta}}
&=
\frac{1-\delta+\theta (\tilde{k}^{ss})^{\theta-1} (\tilde{h}^{ss})^{1-\theta}}{(1+\gamma_n)(1+\gamma_z)}, \\
\frac{\psi}{1-\tilde{h}^{ss}}
&=
\frac{(1-\theta)(\tilde{k}^{ss})^\theta (\tilde{h}^{ss})^{-\theta}}
{\tilde{c}^{ss}}.
\end{aligned}
$$

The first equation is the steady-state resource constraint, the second equation is the steady-state Euler equation, and the third equation is the steady-state labor first-order condition.

We apply Newton's method to solve for the steady-state values
$$
\tilde{k}^{ss}, \qquad \tilde{h}^{ss}, \qquad \tilde{c}^{ss}.
$$


In [33]:
# -------------------------------------------------------
# Steady-state system for the detrended model
# Unknowns:
#   x[1] = k_ss
#   x[2] = h_ss
#   x[3] = c_ss
# Input:
#   beta_tilde : effective discount factor after detrending
#   gamma_n    : population growth rate
#   gamma_z    : technology growth rate
# Output:
#   3 equations evaluated at (k_ss, h_ss, c_ss)
# -------------------------------------------------------
function steady_state_system(x, beta_tilde, psi, theta, delta, gamma_n, gamma_z)
    k = x[1]
    h = x[2]
    c = x[3]
    G = (1.0 + gamma_n) * (1.0 + gamma_z)

    if k <= 0.0 || c <= 0.0 || h <= 0.0 || h >= 1.0
        return [NaN, NaN, NaN]
    end

    F1 = c - k^theta * h^(1.0 - theta) + (G - (1.0 - delta)) * k

    F2 = 1.0 / beta_tilde -
         (1.0 - delta + theta * k^(theta - 1.0) * h^(1.0 - theta)) / G

    F3 = ((1.0 - theta) * k^theta * h^(-theta)) / c -
         psi / (1.0 - h)

    return [F1, F2, F3]
end

# -------------------------------------------------------
# Solve the steady-state system by Newton's method
# Input:
#   beta_tilde, psi, theta, delta, gamma_n, gamma_z : model parameters
#   x0       : initial guess [k_ss, h_ss, c_ss]
#   tol      : convergence tolerance
#   max_iter : maximum number of Newton iterations
#   hstep    : step size for numerical Jacobian
# Output:
#   k_ss, h_ss, c_ss
# -------------------------------------------------------
function solve_steady_state_newton(beta_tilde, psi, theta, delta, gamma_n, gamma_z;
                                   x0=[1.0, 0.3, 0.5],
                                   tol=1e-10, max_iter=100, hstep=1e-6)

    x = Float64.(collect(x0))

    for iter in 1:max_iter
        f(xx) = steady_state_system(xx, beta_tilde, psi, theta, delta, gamma_n, gamma_z)
        F = f(x)

        if any(!isfinite, F)
            error("Newton method entered an infeasible region. Try a different initial guess.")
        end

        if maximum(abs.(F)) < tol
            println("Steady state converged in ", iter, " iterations.")
            return x[1], x[2], x[3]
        end

        J = numerical_jacobian(f, x; h=hstep)
        step = J \ F

        alpha = 1.0
        success = false

        for ls_iter in 1:20
            x_new = x - alpha * step
            k_new, h_new, c_new = x_new

            if k_new > 0.0 && c_new > 0.0 && 0.0 < h_new < 1.0
                F_new = f(x_new)
                if all(isfinite, F_new) && maximum(abs.(F_new)) < maximum(abs.(F))
                    x = x_new
                    success = true
                    break
                end
            end

            alpha *= 0.5
        end

        if !success
            error("Newton method failed to find a feasible update. Try a different initial guess.")
        end
    end

    error("Steady-state Newton solver did not converge within max_iter.")
end


solve_steady_state_newton (generic function with 1 method)

In [35]:
k_ss, h_ss, c_ss = solve_steady_state_newton(
    beta_tilde, psi, theta, delta, gamma_n, gamma_z;
    x0=[1.5, 0.3, 0.4],
    tol=1e-10,
    max_iter=100,
    hstep=1e-6
)
println("steady-state labor h_ss = ", h_ss)
println("steady-state capital k_ss = ", k_ss)
println("steady-state consumption c_ss = ", c_ss)


UndefVarError: UndefVarError: `numerical_jacobian` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.

### Compute optimal labor supply $\tilde{h}^*$ for each $(\tilde{k}, z, \tilde{k}')$ on the grid

We know that the optimal labor supply $\tilde{h}^*$ satisfies the first-order condition
$$
\frac{(1-\theta)\tilde{k}^{\theta} z^{1-\theta}\tilde{h}^{-\theta}}
{\tilde{k}^{\theta}(z\tilde{h})^{1-\theta} - (1+\gamma_n)(1+\gamma_z)\tilde{k}' + (1-\delta)\tilde{k}}
=
\frac{\psi}{1-\tilde{h}}.
$$

For each triple $(\tilde{k}, z, \tilde{k}')$ on the grid, we can solve the above equation for $\tilde{h}$ using numerical root-finding methods, and store the optimal labor supply $\tilde{h}^*(\tilde{k}, z, \tilde{k}')$ in a 3D array.

Write $G(\tilde{h}; \tilde{k}, z, \tilde{k}')$ as:
$$
G(\tilde{h}; \tilde{k}, z, \tilde{k}')
=
\frac{(1-\theta)\tilde{k}^{\theta} z^{1-\theta}\tilde{h}^{-\theta}}
{\tilde{k}^{\theta}(z\tilde{h})^{1-\theta} - (1+\gamma_n)(1+\gamma_z)\tilde{k}' + (1-\delta)\tilde{k}}
-
\frac{\psi}{1-\tilde{h}}.
$$

Use numerical root-finding methods to solve $G(\tilde{h}; \tilde{k}, z, \tilde{k}') = 0$ for $\tilde{h}$, and store the solution as $\tilde{h}^*(\tilde{k}, z, \tilde{k}')$.

If the solution is infeasible:
$$
\tilde{c} = \tilde{k}^\theta (z \tilde{h}^*)^{1-\theta} - (1+\gamma_n)(1+\gamma_z)\tilde{k}' + (1-\delta)\tilde{k} \le 0, \\
\tilde{h}^* < 0, \quad \text{or} \quad \tilde{h}^* > 1,
$$
Return an NaN value for $\tilde{h}^*(\tilde{k}, z, \tilde{k}')$ to indicate that this choice is infeasible.


In [30]:
# -------------------------------------------------------
# Check whether the static problem is feasible
# Since c(h) is increasing in h, it is enough to check
# whether consumption is positive near h = 1
# Input:
#   k       : current capital
#   z       : current productivity
#   kp      : next-period capital
#   theta   : capital share
#   delta   : depreciation rate
#   gamma_n : population growth rate
#   gamma_z : technology growth rate
#   hmax    : maximum labor level (default: 1.0)
# Output:
#   true if the static problem exists feasible interior solution for c* >= 0 & 0 <= h <= 1
# -------------------------------------------------------
function is_feasible_static_problem(k, z, kp, theta, delta, gamma_n, gamma_z; hmax=1.0)
    G = (1.0 + gamma_n) * (1.0 + gamma_z)
    cmax = k^theta * (z * hmax)^(1.0 - theta) - G * kp + (1.0 - delta) * k
    return cmax > 0.0
end

# -------------------------------------------------------
# Labor FOC
# Input:
#   h       : labor
#   k       : current capital
#   z       : current productivity
#   kp      : next-period capital
#   psi     : leisure weight
#   theta   : capital share
#   delta   : depreciation rate
#   gamma_n : population growth rate
#   gamma_z : technology growth rate
# Output:
#   value of the first-order condition
# -------------------------------------------------------
function labor_foc(h, k, z, kp, psi, theta, delta, gamma_n, gamma_z)
    G = (1.0 + gamma_n) * (1.0 + gamma_z)
    c = k^theta * (z * h)^(1.0 - theta) - G * kp + (1.0 - delta) * k

    if c <= 0.0 || h <= 0.0 || h >= 1.0
        return NaN
    end

    lhs = (1.0 - theta) * k^theta * z^(1.0 - theta) * h^(-theta) / c
    rhs = psi / (1.0 - h)

    return lhs - rhs
end

# -------------------------------------------------------
# Solve for h*(k, z, k') by damped Newton's method
# Input:
#   k, z, kp : current capital, productivity, next-period capital
#   psi, theta, delta, gamma_n, gamma_z : model parameters
#   h0       : initial guess for labor
#   tol      : tolerance for root-finding
#   max_iter : maximum number of Newton iterations
#   hstep    : step size for numerical derivative
#   alpha    : damping factor for Newton update
# Output:
#   h_star   : optimal labor supply
# -------------------------------------------------------
function solve_h_star(k, z, kp, psi, theta, delta, gamma_n, gamma_z;
                      h0=0.3, tol=1e-8, max_iter=100, hstep=1e-6, alpha=0.3)

    if !is_feasible_static_problem(k, z, kp, theta, delta, gamma_n, gamma_z)
        return NaN
    end

    h = h0

    for iter in 1:max_iter
        f(hh) = labor_foc(hh, k, z, kp, psi, theta, delta, gamma_n, gamma_z)

        f_val = f(h)

        if !isfinite(f_val)
            return NaN
        end

        if abs(f_val) < tol
            break
        end

        fp_val = numerical_derivative(f, h; h=hstep)

        if !isfinite(fp_val) || abs(fp_val) < 1e-12
            return NaN
        end

        h_new = h - f_val / fp_val
        h_new = min(max(h_new, 1e-6), 1.0 - 1e-6)
        h = alpha * h_new + (1.0 - alpha) * h
    end

    G = (1.0 + gamma_n) * (1.0 + gamma_z)
    c_final = k^theta * (z * h)^(1.0 - theta) - G * kp + (1.0 - delta) * k
    f_final = labor_foc(h, k, z, kp, psi, theta, delta, gamma_n, gamma_z)

    if isfinite(c_final) && c_final > 0.0 && isfinite(f_final) && abs(f_final) < tol
        return h
    else
        return NaN
    end
end

# -------------------------------------------------------
# Compute h_star on the full grid using damped Newton's method
# Input:
#   k_grid, z_grid : grids for capital and productivity
#   psi, theta, delta, gamma_n, gamma_z : model parameters
#   h0       : initial guess for labor
#   tol      : tolerance for root-finding
#   max_iter : maximum number of Newton iterations
#   hstep    : step size for numerical derivative
#   alpha    : damping factor for Newton update
# Output:
#   h_star[ik, iz, ikp] = h*(k, z, k')
# -------------------------------------------------------
function compute_h_star(k_grid, z_grid, psi, theta, delta, gamma_n, gamma_z;
                        h0=0.3, tol=1e-8, max_iter=100, hstep=1e-6, alpha=0.3)

    n_k = length(k_grid)
    n_z = length(z_grid)

    h_star = fill(NaN, n_k, n_z, n_k)

    for ik in 1:n_k
        k = k_grid[ik]

        for iz in 1:n_z
            z = z_grid[iz]
            h_init = h0

            for ikp in 1:n_k
                kp = k_grid[ikp]

                h_guess = solve_h_star(
                    k, z, kp, psi, theta, delta, gamma_n, gamma_z;
                    h0=h_init, tol=tol, max_iter=max_iter, hstep=hstep, alpha=alpha
                )

                h_star[ik, iz, ikp] = h_guess

                if isfinite(h_guess)
                    h_init = h_guess
                end
            end
        end
    end

    return h_star
end


compute_h_star (generic function with 1 method)

Let
$$
\tilde{k}_i \in \{\tilde{k}_1,\dots,\tilde{k}_{n_k}\},
\qquad
z_j \in \{z_1,\dots,z_{n_z}\},
$$

Let $\Pi_z(z,z')$ denote the transition probability from current productivity state $z$ to next-period productivity state $z'$.

Compute the optimal labor supply $\tilde{h}^*(\tilde{k}_i, z_j, \tilde{k}_p)$ for each triple $(\tilde{k}_i, z_j, \tilde{k}_p)$ on the grid, and store the results in a 3D array $\tilde{h}^*_{i,j,p}$.


In [31]:
# -------------------------------------------------------
# Compute h* using Newton's method
# Input:
#   k_grid, z_grid : grids for capital and productivity
#   Other parameters as before
# Output:
#   h_star_grid[ik, iz, ikp] = h*(k, z, k')
# -------------------------------------------------------
h_star = compute_h_star(
    k_grid, z_grid, psi, theta, delta, gamma_n, gamma_z;
    h0=0.5, tol=1e-8, max_iter=100, hstep=1e-6, alpha=0.3
);


UndefVarError: UndefVarError: `k_grid` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Substituting $\tilde{h}^*$ into the Bellman equation, the problem becomes
$$
\begin{aligned}
V(\tilde{k},z)
=
\max_{\tilde{k}'}
\quad
\Biggl\{
&
\frac{
\left[
\tilde{c}^*(\tilde{k},z,\tilde{k}')
\left(1-\tilde{h}^*(\tilde{k},z,\tilde{k}')\right)^\psi
\right]^{1-\sigma}
}{1-\sigma}
+
\tilde{\beta}
\sum_{z' \in Z}
\Pi_z(z,z')\,V(\tilde{k}',z')
\Biggr\}, \\
\text{s.t.} \quad
& \tilde{c}^*(\tilde{k},z,\tilde{k}')
=
\tilde{k}^\theta
\left(z\,\tilde{h}^*(\tilde{k},z,\tilde{k}')\right)^{1-\theta}
-
(1+\gamma_n)(1+\gamma_z)\tilde{k}'
+
(1-\delta)\tilde{k}, \\
& z' \mid z \sim P_z(z,z'), \\
& \tilde{k}' \ge 0, \\
& \tilde{c}^*(\tilde{k},z,\tilde{k}') > 0,
\end{aligned}
$$

Howard policy function iteration speeds up the solution by separating policy improvement from policy evaluation.

1. **Initialize**

- Set an initial value function $V_0(\tilde{k},z)$.
- Set an initial policy function for next-period capital
  $$
  g_k^0(\tilde{k},z).
  $$

2. **Policy Improvement**

- Given the current value function $V_n(\tilde{k},z)$, solve
  $$
  g_k^{n+1}(\tilde{k},z)
  =
  \arg\max_{\tilde{k}'}
  \Biggl\{
  \frac{
  \left[
  \tilde{c}^*(\tilde{k},z,\tilde{k}')
  \left(1-\tilde{h}^*(\tilde{k},z,\tilde{k}')\right)^\psi
  \right]^{1-\sigma}
  }{1-\sigma}
  +
  \tilde{\beta}
  \sum_{z' \in Z}
  \Pi_z(z,z')\,V_n(\tilde{k}',z')
  \Biggr\},
  $$
  subject to
  $$
  \tilde{k}' \ge 0,
  \qquad
  \tilde{c}^*(\tilde{k},z,\tilde{k}') > 0.
  $$

- Once $g_k^{n+1}(\tilde{k},z)$ is determined, recover labor from
  $$
  g_h^{n+1}(\tilde{k},z)
  =
  \tilde{h}^*(\tilde{k},z,g_k^{n+1}(\tilde{k},z)).
  $$

3. **Policy Evaluation**

- Hold the policy functions fixed at
  $$
  g_k^{n+1}(\tilde{k},z), \qquad g_h^{n+1}(\tilde{k},z).
  $$
- For each state $(\tilde{k},z)$, compute the implied consumption
  $$
  \tilde{c}
  =
  \tilde{k}^\theta \bigl(z\,g_h^{n+1}(\tilde{k},z)\bigr)^{1-\theta}
  -
  (1+\gamma_n)(1+\gamma_z)g_k^{n+1}(\tilde{k},z)
  +
  (1-\delta)\tilde{k}.
  $$
- Update the value function repeatedly under the fixed policy:
  $$
  V_{m+1}(\tilde{k},z)
  =
  \frac{
  \left[
  \tilde{c}\,(1-g_h^{n+1}(\tilde{k},z))^\psi
  \right]^{1-\sigma}
  }{1-\sigma}
  +
  \tilde{\beta}
  \sum_{z' \in Z}
  \Pi_z(z,z')\,V_m(g_k^{n+1}(\tilde{k},z),z').
  $$
- Perform this policy evaluation step for several iterations while keeping the policy fixed.

4. **Convergence Check**

- After the policy improvement step, record the number of grid points at which the policy function changes:
  $$
  n_{\text{change}}
  =
  \#\left\{
  (\tilde{k},z)
  :
  g_k^{n+1}(\tilde{k},z) \neq g_k^n(\tilde{k},z)
  \right\}.
  $$
- If
  $$
  n_{\text{change}} = 0,
  $$
  then the policy function is stable on the grid, and the algorithm has converged.
- Otherwise, set
  $$
  V_n \leftarrow V_{n+1},
  \qquad
  g_k^n \leftarrow g_k^{n+1},
  $$
  and repeat Steps 2--4.


Howard policy function iteration is faster than standard value function iteration because the maximization problem is now one-dimensional in $\tilde{k}'$, while labor is recovered from the static first-order condition.


In [32]:
# -------------------------------------------------------
# Flow utility function
# Input:
#   c     : detrended consumption
#   h     : labor
#   psi   : leisure weight
#   sigma : coefficient of relative risk aversion
# Output:
#   period utility
# -------------------------------------------------------
function flow_utility(c, h, psi, sigma)
    if c <= 0.0 || h <= 0.0 || h >= 1.0
        return -Inf
    end

    return ((c * (1.0 - h)^psi)^(1.0 - sigma)) / (1.0 - sigma)
end

# -------------------------------------------------------
# Solve the detrended growth model by Howard policy iteration
# using the precomputed labor supply h_star
# State variables:
#   (k_tilde, z)
# Control variable:
#   k_tilde_next
# Labor choice:
#   recovered from h_star[ik, iz, ikp]
# Input:
#   beta_tilde, psi, sigma, gamma_n, gamma_z, theta, delta
#   k_grid, z_grid, Pz, h_star
#   tol, max_iter, howard_iter
# Output:
#   V          : value function
#   pol_kp     : policy function for next-period capital
#   pol_h      : policy function for labor
# -------------------------------------------------------
function solve_pfi_howard(beta_tilde, psi, sigma, gamma_n, gamma_z, theta, delta,
                          k_grid, z_grid, Pz, h_star;
                          tol=1e-6, max_iter=1000, howard_iter=20)

    n_k = length(k_grid)
    n_z = length(z_grid)
    G = (1.0 + gamma_n) * (1.0 + gamma_z)

    V = zeros(n_k, n_z)
    V_new = similar(V)

    pol_kp_idx = ones(Int, n_k, n_z)
    pol_kp = zeros(n_k, n_z)
    pol_h = zeros(n_k, n_z)

    for iter in 1:max_iter
        pol_kp_idx_old = copy(pol_kp_idx)

        for iz in 1:n_z
            z = z_grid[iz]

            for ik in 1:n_k
                k = k_grid[ik]

                best_val = -Inf
                best_kp_idx = 1

                for ikp in 1:n_k
                    kp = k_grid[ikp]
                    h = h_star[ik, iz, ikp]

                    if !isfinite(h)
                        continue
                    end

                    c = k^theta * (z * h)^(1.0 - theta) - G * kp + (1.0 - delta) * k
                    u = flow_utility(c, h, psi, sigma)

                    if !isfinite(u)
                        continue
                    end

                    EV = 0.0
                    for izp in 1:n_z
                        EV += Pz[iz, izp] * V[ikp, izp]
                    end

                    val = u + beta_tilde * EV

                    if val > best_val
                        best_val = val
                        best_kp_idx = ikp
                    end
                end

                if best_val == -Inf
                    error("No feasible choice found at state (ik=$ik, iz=$iz).")
                end

                pol_kp_idx[ik, iz] = best_kp_idx
            end
        end

        n_change = count(pol_kp_idx .!= pol_kp_idx_old)
        println("Howard iteration = ", iter, ", policy changes = ", n_change)

        for hiter in 1:howard_iter
            for iz in 1:n_z
                z = z_grid[iz]

                for ik in 1:n_k
                    k = k_grid[ik]
                    ikp = pol_kp_idx[ik, iz]
                    kp = k_grid[ikp]
                    h = h_star[ik, iz, ikp]

                    c = k^theta * (z * h)^(1.0 - theta) - G * kp + (1.0 - delta) * k
                    u = flow_utility(c, h, psi, sigma)

                    EV = 0.0
                    for izp in 1:n_z
                        EV += Pz[iz, izp] * V[ikp, izp]
                    end

                    V_new[ik, iz] = u + beta_tilde * EV
                end
            end

            V .= V_new
        end

        if iter > 1 && n_change == 0
            for iz in 1:n_z
                for ik in 1:n_k
                    ikp = pol_kp_idx[ik, iz]
                    pol_kp[ik, iz] = k_grid[ikp]
                    pol_h[ik, iz] = h_star[ik, iz, ikp]
                end
            end

            println("Howard policy iteration converged in ", iter, " iterations.")
            return V, pol_kp, pol_h
        end
    end

    error("Howard policy iteration did not converge within max_iter.")
end


solve_pfi_howard (generic function with 1 method)